In [10]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import polars as pl

# --- CONFIG ---
emb_model = "qwen3-8b-with-instruction"
path = f'../../data/raw/{emb_model}/embeddings_raw.parquet'
n_dims = 128

# Read in Data
item_embeddings = pl.read_parquet(path)

# Get the number of embeddings in item_embeddings
n_embeddings = len([col for col in item_embeddings.columns if col.startswith("emb")])

# Use GPU if available; for model-train speed
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# Convert embeddings columns to a torch tensor
X_train = item_embeddings.select(pl.col('^emb.*$')).to_numpy()
X_tensor = torch.tensor(X_train, dtype=torch.float32).to(device)

# Define Architecture
class PsychometricAE(nn.Module):
    def __init__(self, input_dim=n_embeddings, bottleneck_dim=n_dims):
        super().__init__()
        # Encoder: Funneling down to the signal
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 1024),
            nn.ReLU(),
            nn.Linear(1024, bottleneck_dim) # Linear bottleneck to preserve distance logic
        )
        # Decoder: Reconstructing to ensure information was kept
        self.decoder = nn.Sequential(
            nn.Linear(bottleneck_dim, 1024),
            nn.ReLU(),
            nn.Linear(1024, input_dim)
        )

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded

# Training Setup
model = PsychometricAE().to(device)
criterion = nn.MSELoss().to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training Loop 
print("Training Autoencoder...")
for epoch in range(500):
    optimizer.zero_grad()
    outputs = model(X_tensor)
    loss = criterion(outputs, X_tensor)
    loss.backward()
    optimizer.step()
    
    if (epoch + 1) % 20 == 0:
        print(f"Epoch [{epoch+1}/500], Loss: {loss.item():.6f}")

# Extract Compressed Features
with torch.no_grad():
    compressed_vectors = model.encoder(X_tensor).cpu().numpy()

column_names = [f"emb_{i+1}" for i in range(n_dims)]

# Return to Polars
embeddings_compressed = (
    item_embeddings.with_columns(
        pl.Series("compressed_embedding", [list(v) for v in compressed_vectors])
    )
    .select([
        pl.col("item"),
        pl.col("compressed_embedding").list.to_struct(
            fields=column_names # Names the columns
        )
    ])
    .unnest("compressed_embedding")
)

print("Success! Compressed features added to dataframe.")

Training Autoencoder...
Epoch [20/500], Loss: 0.000128
Epoch [40/500], Loss: 0.000122
Epoch [60/500], Loss: 0.000112
Epoch [80/500], Loss: 0.000102
Epoch [100/500], Loss: 0.000093
Epoch [120/500], Loss: 0.000084
Epoch [140/500], Loss: 0.000078
Epoch [160/500], Loss: 0.000072
Epoch [180/500], Loss: 0.000068
Epoch [200/500], Loss: 0.000064
Epoch [220/500], Loss: 0.000061
Epoch [240/500], Loss: 0.000059
Epoch [260/500], Loss: 0.000056
Epoch [280/500], Loss: 0.000054
Epoch [300/500], Loss: 0.000053
Epoch [320/500], Loss: 0.000051
Epoch [340/500], Loss: 0.000049
Epoch [360/500], Loss: 0.000048
Epoch [380/500], Loss: 0.000046
Epoch [400/500], Loss: 0.000045
Epoch [420/500], Loss: 0.000044
Epoch [440/500], Loss: 0.000043
Epoch [460/500], Loss: 0.000042
Epoch [480/500], Loss: 0.000042
Epoch [500/500], Loss: 0.000040
Success! Compressed features added to dataframe.


In [11]:
# Save the weights to disk
torch.save(model.state_dict(), '../../models/psychometric_ae_weights.pt')
print("Model saved successfully!")


Model saved successfully!


In [12]:
import os

# Read in item correlations data and item list with transformers features
item_cors_df = pl.read_parquet(f"../../data/processed/item_correlations_with_cross.parquet")
item_list_df = pl.read_parquet(f"../../data/processed/item_list_sentiment.parquet")

# Join compressed embeddings into item_list_df to combine features
items_embedded = item_list_df.join(
    embeddings_compressed, 
    on='item',
    how='left'
)

# Create Subfolders if they do not exist yet
os.makedirs(f"../../data/clustered_embeddings", exist_ok=True)
os.makedirs(f"../../data/clustered_embeddings/{emb_model}", exist_ok=True)

# Save Item list with all features to disk
items_embedded.write_parquet(f"../../data/clustered_embeddings/{emb_model}/item_list_emb_{n_dims}.parquet")

In [13]:
# Join embeddings per item into the pairwise item correlations dataframe
item_cors_df = item_cors_df.join(
    embeddings_compressed, right_on='item', left_on='Parameter1', how='left', suffix='_item1'
).join(
    embeddings_compressed, right_on='item', left_on='Parameter2', how='left', suffix='_item2'
)

# Exclude mean, sd, min, max from the item_list_df to prevent leaks
item_list_df = item_list_df.select(
        pl.exclude(['mean', 'sd', 'max', 'min', 'item_right'])
)


In [14]:
import polars.selectors as cs
import numpy as np
from numpy.linalg import norm

# Define arrays of embeddings columns per item
cols_1 = [f"emb_{i+1}" for i in range(n_dims)]
cols_2 = [f"emb_{i+1}_item2" for i in range(n_dims)]

# 1. Product Features (Element-wise)
# Basically cosine similarity using Point estimates instead of vectors
product_features = [
    (pl.col(c1) * pl.col(c2)).alias(f"prod_{i+1}") 
    for i, (c1, c2) in enumerate(zip(cols_1, cols_2))
]

# 2. Global Cosine Similarity 
# Cosine = Dot Product / (Norm A * Norm B)
dot_product = pl.sum_horizontal([pl.col(c1) * pl.col(c2) for c1, c2 in zip(cols_1, cols_2)])
norm_a = pl.sum_horizontal([pl.col(c)**2 for c in cols_1]).sqrt()
norm_b = pl.sum_horizontal([pl.col(c)**2 for c in cols_2]).sqrt()

# Add in global similarity feature and exclude the raw item embeddings from final df
df_final = item_cors_df.with_columns(product_features).with_columns(
    global_sim = dot_product / (norm_a * norm_b)
).select(
    ~cs.starts_with("emb")
)

# Rename columns in item_list to have the suffix "_item1" or "_item2"
item_list_v1 = item_list_df.rename(lambda column_name: column_name + "_item1")
item_list_v2 = item_list_df.rename(lambda column_name: column_name + "_item2")

# Join in these item_list versions into final df
# Good for readability of feature importances while modelling
df_final = df_final.join(
    item_list_v1, 
    left_on='Parameter1',
    right_on='item_item1',
    how='left',
).join(
    item_list_v2,
    left_on='Parameter2',
    right_on='item_item2',
    how='left',
)

In [15]:
# Overview of the dataframe
df_final.head()

Parameter1,Parameter2,r,pair_negative,pair_positive,contradiction,entail,logic_neutral,similarity,thematic_intensity,logical_friction,sentiment_balance,prod_1,prod_2,prod_3,prod_4,prod_5,prod_6,prod_7,prod_8,prod_9,prod_10,prod_11,prod_12,prod_13,prod_14,prod_15,prod_16,prod_17,prod_18,prod_19,prod_20,prod_21,prod_22,prod_23,prod_24,prod_25,…,prod_117,prod_118,prod_119,prod_120,prod_121,prod_122,prod_123,prod_124,prod_125,prod_126,prod_127,prod_128,global_sim,sent_positive_item1,sent_neutral_item1,sent_negative_item1,emo_neutral_item1,emo_surprise_item1,emo_joy_item1,emo_fear_item1,emo_anger_item1,emo_sadness_item1,emo_disgust_item1,top_sentiment_item1,top_emotion_item1,sent_positive_item2,sent_neutral_item2,sent_negative_item2,emo_neutral_item2,emo_surprise_item2,emo_joy_item2,emo_fear_item2,emo_anger_item2,emo_sadness_item2,emo_disgust_item2,top_sentiment_item2,top_emotion_item2
str,str,f64,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,…,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,str
"""Do you often long for exciteme…","""Do you often need understandin…",0.201111,0.011093,0.679199,0.999023,0.000023,0.001016,0.016022,3.6862e-7,0.016006,0.668106,0.000699,0.005163,0.010033,-0.001857,0.000028,0.004854,0.003163,0.0122,0.000227,0.006445,0.006042,0.00717,0.00375,0.001434,0.007706,-0.000968,0.000962,0.000525,0.000302,-0.000169,0.004612,-0.000116,-0.000105,0.003678,0.000024,…,0.000476,0.0012,-0.001263,0.004908,0.008891,0.000202,0.028448,0.000391,0.003286,0.000063,0.000005,0.001203,0.827537,0.783163,0.205401,0.011436,0.784398,0.119589,0.032254,0.030117,0.02025,0.01002,0.003373,"""positive""","""neutral""",0.597145,0.379198,0.023657,0.749233,0.072289,0.127554,0.005687,0.017466,0.013736,0.014034,"""positive""","""neutral"""
"""Do you often long for exciteme…","""Do you stop and think things o…",-0.013197,0.02562,0.254639,1.0,0.000011,0.000136,0.006691,7.2185e-8,0.006691,0.229019,0.0003,0.007385,0.004294,-0.005387,0.000049,-0.002234,0.005276,0.012606,-0.00005,0.010128,0.011414,0.006633,0.00726,0.000376,0.008259,-0.000907,-0.001055,0.000378,0.002084,-0.000371,0.000328,0.000287,-0.000189,0.003623,0.000002,…,0.000339,-0.001162,0.00093,0.007455,0.008576,0.000654,0.027768,-0.000259,-0.001229,0.002194,0.000003,0.0006,0.792505,0.783163,0.205401,0.011436,0.784398,0.119589,0.032254,0.030117,0.02025,0.01002,0.003373,"""positive""","""neutral""",0.021467,0.800833,0.1777,0.062341,0.01233,0.002852,0.072813,0.784729,0.011009,0.053926,"""neutral""","""anger"""
"""Do you often long for exciteme…","""If you say you will do somethi…",-0.025196,0.038971,0.341797,0.012039,0.000207,0.987793,0.002369,4.8940e-7,0.000029,0.302826,0.002146,0.007903,0.011022,-0.003976,-0.000047,0.000634,0.00396,0.013697,-0.000016,0.012709,0.011117,0.010729,0.008807,-0.001681,0.00999,-0.002559,-0.001191,0.00016,0.00209,0.000069,0.000075,0.000034,-0.000206,0.001153,0.000052,…,-0.000369,0.001231,0.000655,0.00532,0.00406,0.001572,0.033391,-0.000996,0.003082,-0.001016,0.000009,0.000748,0.800969,0.783163,0.205401,0.011436,0.784398,0.119589,0.032254,0.030117,0.02025,0.01002,0.003373,"""positive""","""neutral""",0.052955,0.69619,0.250854,0.51845,0.006533,0.004173,0.020429,0.382281,0.01385,0.054285,"""neutral""","""neutral"""
"""Do you often long for exciteme…","""Do your moods go up and down?""",0.135682,0.02153,0.2771,0.974609,0.001231,0.024017,0.020416,0.000025,0.019898,0.255569,0.000692,0.010295,0.009962,-0.001907,0.000046,0.001845,0.001808,0.010026,0.000205,0.008117,0.007379,0.007317,0.005734,-0.001738,0.008803,0.00084,0.002214,-0.000135,0.002092,0.000516,0.001646,-0.000074,-0.000098,0.002238,0.00004,…,0.000096,0.001745,-0.000474,0.002092,0.011196,0.000455,0.030539,0.000886,0.00338,0.001991,5.4161e-7,0.000098,0.870753,0.783163,0.205401,0.011436,0.784398,0.119589,0.032254,0.030117,0.02025,0.01002,

In [16]:
# Save to disk
df_final.write_parquet(f"../../data/clustered_embeddings/{emb_model}/autoencoded_clusters_{n_dims}.parquet")